# Introduction

# Add root to sys.path

In [11]:
import os
import sys

# Add the root directory to the Python path
module_dir = os.path.abspath('..')
if module_dir not in sys.path:
    sys.path.append(module_dir)
for x in sys.path:
    print(x)

/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/jupyter_debug
/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev
/opt/anaconda3/envs/python312/lib/python312.zip
/opt/anaconda3/envs/python312/lib/python3.12
/opt/anaconda3/envs/python312/lib/python3.12/lib-dynload

/opt/anaconda3/envs/python312/lib/python3.12/site-packages
/opt/anaconda3/envs/python312/lib/python3.12/site-packages/setuptools/_vendor
/Users/hale/PycharmProjects/MathAssertGPT


# Imports

In [ ]:
import torch

from source.shared.neural_network.Head import Head

from source.evaluate_model.get_syntax_deriver import get_syntax_deriver
from source.evaluate_model.ModelEvaluator import get_prompt, get_reply

from Settings import Settings

# Settings

In [13]:
import panel as pn
pn.extension()
pn.config.sizing_mode="stretch_width"

settings = Settings()
settings.view()

Traceback (most recent call last):
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/pyviz_comms/__init__.py", line 341, in _handle_msg
 self._on_msg(msg)
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/panel/viewable.py", line 491, in _on_msg
 patch = manager.assemble(msg)
 ^^^^^^^^^^^^^^^^^^^^^
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/panel/models/comm_manager.py", line 25, in assemble
 header = msg['header']
 ~~~^^^^^^^^^^
KeyError: 'header'

WidgetBox(max_width=600, sizing_mode='stretch_width')
    [0] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='Limit count', sizing_mode='stretch_width', value=40000)
    [2] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='n_embd', sizing_mode='stretch_width', value=1000)
        [2] IntInput(name='n_head', sizing_mode='stretch_width', value=10)
        [3] IntInput(name='block_size', sizing_mode='stretch_width', value=150)
        [4] FloatInput(name='dropout', sizing_mode='stretch_width', value=0.2)
        [5] IntInput(name='n_layer', sizing_mode='stretch_width', value=10)
    [3] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] FloatInput(name='learning_rate', sizing_mode='stretch_width', value=0.0001)
    [4] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] LiteralInput(name='mmx_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [2] LiteralInput(name='corpus01_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [3] LiteralInput(name='corpus_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [4] LiteralInput(name='model_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
    [5] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')

Traceback (most recent call last):
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/pyviz_comms/__init__.py", line 341, in _handle_msg
 self._on_msg(msg)
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/panel/viewable.py", line 491, in _on_msg
 patch = manager.assemble(msg)
 ^^^^^^^^^^^^^^^^^^^^^
 File "/opt/anaconda3/envs/python312/lib/python3.12/site-packages/panel/models/comm_manager.py", line 25, in assemble
 header = msg['header']
 ~~~^^^^^^^^^^
KeyError: 'header'

# Debug Head

In [14]:
n_embd = 7
n_head = 2
head_size = n_embd // n_head
block_size = 1
dropout = 0.2
head = Head(head_size, n_embd, block_size, dropout)
print(f'head_size={head_size}')

head_size=3


In [15]:
batch_size = 1
time_step = 5
x=torch.randn(batch_size, time_step, n_embd)
print(f'x.shape={x.shape}')
print(f'x={x}')
head(x=x)

x.shape=torch.Size([1, 5, 7])
x=tensor([[[ 0.5046, -1.6624,  0.1070,  0.5973, -0.3970, -0.2803, -0.5486],
         [ 0.8894,  1.2049,  0.4468, -1.3005, -0.8114, -1.2359,  0.1790],
         [ 0.0372,  0.5308,  0.1394, -1.5270,  0.2241, -0.0156, -0.4852],
         [ 1.9646,  0.8013,  1.2190,  0.1674, -0.1877,  0.4140, -0.4147],
         [ 1.1922, -0.6017,  0.3984,  0.7112,  1.3061, -1.0061,  1.0106]]])


tensor([[[-0.0023, -0.2561, -0.2693],
         [ 0.1088, -0.2176,  0.1410],
         [ 0.0484, -0.1156, -0.2034],
         [ 0.1223, -0.2938, -0.1473],
         [ 0.0141, -0.2653, -0.1976]]], grad_fn=<UnsafeViewBackward0>)

# Debug syntax_deriver

In [16]:
def unit_test_evaluate_model():
    terminal_token = '<|over|>'
    syntax_deriver = get_syntax_deriver(corpus_folder_path=settings.corpus_folder_path)
    prompt = get_prompt()
    # wff = '( E. x A. y ( y e. x <-> E. x ( x e. w /\\ A e. y ) ) <-> E. y A. x ( x e. y <-> ( A F x <-> E. y ( y e. z /\\ E. x ( x e. w /\\ A F y ) ) ) ) )'
    # wff = '( ( ( ( ( ps -> ps ) -> ( -. -. ps -> -. ps ) ) -> -. ps ) -> ps ) -> ph ) -> ps ) -> ( ( ps /\ ps ) -> ( -. ps -> -. ps ) ) )'  # this has a wff followed by more tokens
    # wff = '( ( ( ( ( ps -> ps ) -> ( -. -. ps -> -. ps ) ) -> -. ps ) -> ps ) -> ph )'  # this is a wff
    wff = '( A e. V -> [_ A / x ]_ { C } = { [_ [_ A / x ]_ C } )'
    predicted_statement = f'|- {wff} {terminal_token}'
    wff_statement = get_reply(dictum=predicted_statement, terminal_token=terminal_token)
    context = '\n'.join([prompt, predicted_statement])
    syntax_deriver.derive_syntax(statement=wff_statement, context=context)
    is_ok = syntax_deriver.syntaxDerivation is not None
    print(f'is_ok={is_ok}')

unit_test_evaluate_model()

is_ok=False


In [17]:
syntax_deriver = get_syntax_deriver(corpus_folder_path=settings.corpus_folder_path)

In [18]:
f'{syntax_deriver.syntax_deriver_db.show_rule_errors_table()}'

--- rule_errors ---


'None'